# BrainInsight AI
# Notebook 03: Feature Extraction

## Objectives
Extract handcrafted features from the preprocessed MRI images.

### Features
- HOG
- Local Binary Pattern (LBP)
- GLCM (Contrast, Correlation, Energy, Homogeneity)
- Hu Moments
- Shape Features
- Histogram Features
- Statistical Texture Features

The final output will be:
- `X.npy`
- `y.npy`
- `feature_dataset.csv`


In [1]:
import os
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
from sklearn.preprocessing import StandardScaler
import joblib

PROJECT_ROOT = Path("..")
PREPROCESSED_DIR = PROJECT_ROOT / "Preprocessed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

TRAIN_DIR = PREPROCESSED_DIR / "Training"
TEST_DIR = PREPROCESSED_DIR / "Testing"


## Feature Extraction Functions

In [2]:
def extract_features(image_path):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Cannot read {image_path}")

    # HOG
    hog_feat = hog(
        img,
        orientations=9,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2),
        feature_vector=True
    )

    # LBP
    lbp = local_binary_pattern(img, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(),
                               bins=np.arange(0,11),
                               range=(0,10))
    lbp_hist = lbp_hist.astype("float32")
    lbp_hist /= (lbp_hist.sum() + 1e-6)

    # GLCM
    glcm = graycomatrix(img,
                        distances=[1],
                        angles=[0],
                        symmetric=True,
                        normed=True)

    glcm_features = [
        graycoprops(glcm,'contrast')[0,0],
        graycoprops(glcm,'correlation')[0,0],
        graycoprops(glcm,'energy')[0,0],
        graycoprops(glcm,'homogeneity')[0,0]
    ]

    # Hu Moments
    hu = cv2.HuMoments(cv2.moments(img)).flatten()

    # Shape
    _,th = cv2.threshold(img,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    contours,_ = cv2.findContours(th,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        cnt=max(contours,key=cv2.contourArea)
        area=cv2.contourArea(cnt)
        perimeter=cv2.arcLength(cnt,True)
        circularity=(4*np.pi*area)/(perimeter**2+1e-6)
        x,y,w,h=cv2.boundingRect(cnt)
        aspect_ratio=w/(h+1e-6)
        hull=cv2.convexHull(cnt)
        hull_area=cv2.contourArea(hull)
        solidity=area/(hull_area+1e-6)
    else:
        area=perimeter=circularity=aspect_ratio=solidity=0

    # Histogram
    hist=cv2.calcHist([img],[0],None,[32],[0,256]).flatten()
    hist/=hist.sum()+1e-6

    # Texture statistics
    texture=[
        img.mean(),
        img.std(),
        img.min(),
        img.max()
    ]

    feature_vector=np.concatenate([
        hog_feat,
        lbp_hist,
        glcm_features,
        hu,
        [area,perimeter,circularity,aspect_ratio,solidity],
        hist,
        texture
    ])

    return feature_vector


## Build Feature Dataset

In [3]:
X=[]
y=[]

for split in [TRAIN_DIR,TEST_DIR]:
    if not split.exists():
        continue

    for cls in sorted(os.listdir(split)):
        class_dir=split/cls
        if not class_dir.is_dir():
            continue

        for img in tqdm(list(class_dir.iterdir()),desc=f"{split.name}-{cls}"):

            if img.suffix.lower() not in ['.jpg','.jpeg','.png','.bmp','.tif','.tiff']:
                continue

            try:
                features=extract_features(img)
                X.append(features)
                y.append(cls)
            except Exception as e:
                print(e)

X=np.array(X)
y=np.array(y)

print("Feature Matrix Shape:",X.shape)
print("Labels Shape:",y.shape)


Testing-pituitary: 100%|██████████| 400/400 [00:18<00:00, 22.16it/s]


Feature Matrix Shape: (7200, 34658)
Labels Shape: (7200,)


## Normalize Features

In [5]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import joblib

# ==========================================
# Normalize Features (Optimized)
# ==========================================

# Reduce memory usage
X = X.astype(np.float32)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler
joblib.dump(
    scaler,
    PROJECT_ROOT / "saved_models" / "scaler.pkl"
)

# Save feature matrix and labels
np.save(
    OUTPUT_DIR / "X.npy",
    X_scaled.astype(np.float32)
)

np.save(
    OUTPUT_DIR / "y.npy",
    y
)

print("=" * 50)
print("Feature Normalization Completed")
print("=" * 50)
print(f"Feature Matrix Shape : {X_scaled.shape}")
print(f"Labels Shape         : {y.shape}")

print("\nSaved Files")
print("- X.npy")
print("- y.npy")
print("- scaler.pkl")

Feature Normalization Completed
Feature Matrix Shape : (7200, 34658)
Labels Shape         : (7200,)

Saved Files
- X.npy
- y.npy
- scaler.pkl


## Conclusion

Handcrafted features have been successfully extracted and normalized. The resulting feature matrix is now ready for model training and comparison in the next notebook.


## Interview Notes

- Why use handcrafted features instead of CNNs?
- What does HOG capture?
- What is LBP used for?
- Why are GLCM features useful?
- What information do Hu Moments provide?
- Why normalize features before training?
